## Linear Support Vector machine (SVM) - `LinearSVC`

All training code called here.

- Training Time Taken
- Test Accuracy

### Model Parameters

For Linear SVM, the regularization parameter C will be tested at $0.01, 0.1, 1, 10, 100$, with the best value selected using cross-validation.

### Load Shared Library

`autoreload` enables it to reload all changes in `shared_library.py`


In [1]:
%load_ext autoreload
%autoreload 2 

# import everything!!
from shared_library import * 

### Output Details

Results for linear classification are captured in `linear_results.csv`


In [2]:
RESULTS_CSV = f"./results/linear_experiments_results_{yyyymmddhhmm()}.csv"
N_JOBS = 5

### Experiments

#### Attempt 1: To Infinity and Beyond!

Write loops and capture output in `DataFrame` for cleaner display on the notebook. Brute force approach to identify $C$, there should be better way to identify the best $C$...

**Note**: Running a cartesian product of all ImageCounts, Cs and Iterations is a bad idea.

> ```python
> # Let's see how much data is needed, beyond which it is a point of diminishing returns.
> imgs = [500, 1000, 2000, 3000, 4000, 5000]
> cs = [0.01, 0.1, 1, 10, 100]
> iters = [10000]
> experiments = list(product(imgs, iters, cs))
>
> df = pd.DataFrame([linear(ic, itr, c) for ic, itr, c in experiments])
> display(df)
> ```

The above combination generated 30 experiments. At 700 minutes, I was in "The Sunk Cost Fallacy" land. Since I had burnt through so much processing power, I decided to let the program complete. After let it run over 930 minutes ~ 15.5 hours, and had to terminate the program.


#### Attempt 2: Back to Earth


In [8]:
def linear(max_iter, C):

    # per convention X is data, y is labels.
    X, y =  load_all_images_flat(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)

    print(X.shape, X.dtype, X.min(), X.max())

    print(f"Running for max_iter:{max_iter}, C: {C}")

    start_time = perf_counter()
    clf = LinearSVC(loss="hinge", C=C, max_iter=max_iter, verbose=1, random_state=SEED)
    scores = cross_val_score(
        estimator=clf,
        X=X,
        y=y,
        n_jobs=N_JOBS,
    )
    end_time = perf_counter()

    return {
        "C": C,
        "Iterations": max_iter,
        "Mean": round(scores.mean(), 4),
        "Sigma": round(scores.std(), 4),
        "Time": end_time - start_time,
    }

cs = [0.01, 0.1, 1]
iters = [1000]
experiments = list(product(iters, cs))

# init results
rows = []
df = pd.DataFrame()

print(f"Running {len(experiments)} experiments")

Running 3 experiments


In [4]:
from IPython.display import clear_output

for it, c in experiments:
    row = linear(it, c)
    rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)  # flush to disk immediately
    clear_output(wait=True)
    display(df)

print("This experiment was last executed on: ", datetime.datetime.now())

,C,Iterations,Mean,Sigma,Time
0,0.01,500,0.2996,0.0096,3704.773138
1,0.10,500,0.2756,0.0132,9215.018456
2,1.00,500,0.2758,0.0124,4104.535261


This experiment was last executed on:  2026-10-06 22:44:25.300195


### Cross Validation

[`cross_val_score`](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-evaluating-estimator-performance) - Does 5 fold cross validation to find the best parameter for C

$C$ is regularization, lower the $\Delta x$, stronger the regularization, so when adding $\Delta x$ to $w^T x$, the value of $w^T\Delta x$ should be moved a small amount.


In [5]:
# # Iterate over multiple values of Cs, and find the best results
# Cs = [
#     0.01,
#     0.1,
#     1,
#     10,
#     100,
# ]  # 10 and 100 will run, if 0.1 and 1 have a huge convergence

# results = []
# for c in Cs:
#     print("Running for C: ", c, end=" | ")
#     start_time = perf_counter()
#     clf = LinearSVC(loss="hinge", C=c, max_iter=10000, verbose=0, random_state=42)
#     scores = cross_val_score(estimator=clf, X=X, y=labels, n_jobs=-1)
#     end_time = perf_counter()

#     print("Time Taken: ", end_time - start_time)
#     print("-------------")

#     results.append(
#         {
#             "C": c,
#             "Mean": round(scores.mean(), 4),
#             "Average": sum(scores) / len(scores),
#             "Sigma": round(scores.std(), 4),
#             "Time": (end_time - start_time),
#         }
#     )

# linear_results = pd.DataFrame(results)
# display(linear_results)

# print("This experiment was last executed on: ", datetime.datetime.now())

### Cross Validation

`cross_val_score` - Does 5 fold cross validation

Find `C` with the maximum **Mean Accuracy**, and we'll use that to do a `fit` test against test data


In [6]:
# # Find the row of the data with the max mean, we can run into issues for multiple Cs having the same mean, does it matter???
# # going to try this overnight while the CPU runs cycles!
# max_mean = linear_results.iloc[linear_results["Mean"].argmax()]

# # Find the value of C from the row, which has the max mean
# best_c = float(max_mean["C"])

# best_start_time = perf_counter()
# best_model = LinearSVC(loss="hinge", C=best_c, max_iter=100000, verbose=0).fit(
#     images, labels
# )
# best_end_time = perf_counter()

# score = best_model.score(test_images, test_labels)

# print(
#     f"Best C: {best_c} | Score: {score} | Final Train Time: {best_end_time - best_start_time}s"
# )

# print("This experiment was last executed on: ", datetime.datetime.now())